cutehand Python金融量化，2023年09月04日 ，【手把手教你】使用DoWhy做因果推断分析，https://mp.weixin.qq.com/s/TQU4dSKQhxc-ouat-lWGrQ

In [1]:
!pip install qstock
!pip install dowhy

     |████████████████████████████████| 151 kB 67 kB/s eta 0:00:01
     |████████████████████████████████| 151 kB 214 kB/s eta 0:00:01
  Using cached tqdm-4.64.1-py2.py3-none-any.whl (78 kB)
  Using cached jieba-0.42.1.tar.gz (19.2 MB)
     |████████████████████████████████| 292 kB 186 kB/s eta 0:00:01
     |████████████████████████████████| 147 kB 238 kB/s eta 0:00:01
  Using cached plotly-5.18.0-py3-none-any.whl (15.6 MB)
     |████████████████████████████████| 5.3 MB 928 kB/s eta 0:00:01
     |████████████████████████████████| 44 kB 184 kB/s eta 0:00:01
  Using cached soupsieve-2.3.2.post1-py3-none-any.whl (37 kB)
     |████████████████████████████████| 73 kB 512 kB/s eta 0:00:01
  Created wheel for qstock: filename=qstock-1.3.6-py3-none-any.whl size=159219 sha256=149aee62fe74a4301a8b3bc807a6d5d99c4a6b019ebe98a3807cde8d860777ca
  Stored in directory: /Users/wangshuaibo/Library/Caches/pip/wheels/c2/48/ac/a99149976683c53ce380c5be11a6726232afa13a98cb5d5847
  Created wheel for func-time

In [4]:
!pip install pywencai

In [8]:
!pip install pandas_ta

     |████████████████████████████████| 115 kB 532 kB/s eta 0:00:01
  Created wheel for pandas-ta: filename=pandas_ta-0.3.14b0-py3-none-any.whl size=218923 sha256=3361ff3ecf66536adf438f5c76443cea21c0402c691e3b2cb31c69117e459ac6
  Stored in directory: /Users/wangshuaibo/Library/Caches/pip/wheels/a9/e4/78/4fe191ef4161aec0b49f6bba1bdf493e55a93b16f1af6cb3d0
Successfully built pandas-ta


In [10]:
!pip install dataclasses

In [17]:
!pip install nbformat

     |████████████████████████████████| 178 kB 244 kB/s eta 0:00:01
     |████████████████████████████████| 56 kB 668 kB/s eta 0:00:01
  Using cached attrs-22.2.0-py3-none-any.whl (60 kB)
     |████████████████████████████████| 68 kB 448 kB/s eta 0:00:01


In [20]:
pip install ipykernel

Note: you may need to restart the kernel to use updated packages.


In [1]:
import qstock as qs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import dowhy
from dowhy import CausalModel
import pandas_ta as ta

In [9]:
#获取沪深 300 指数数据
df = qs.get_data("hs300", start="20130901", end="20220901")

# 计算日收益率和跳空高开（Gap Up）
df["daily_return"] = df["close"].pct_change() * 100 #日收益率
df["gap_up"] = (df["open"] - df["close"].shift(1)) > 0 #跳空高开

# 特征提取
df["mean_lagged_return"] = df["daily_return"].rolling(5).mean().shift(1)
df["var_lagged_return"] = df["daily_return"].rolling(5).var().shift(1)
df["volume%"] = df["volume"].pct_change()
df["ma_20"] = df.close.rolling(20).mean()
df["ma_close"] = df["ma_20"].shift(1)/df.close.shift(1)-1
#df["macd"] = ta.macd("close").iloc[:,0].shift(1)
#df["rsi"] = ta.rsi().shift(1)

#删除NaN
df.dropna(inplace=True)

In [10]:
qs.line(df["daily_return"])

In [11]:
qs.box(x="gap_up", y="daily_return", data=df)

In [12]:
# 定义共同因素和因果模型
covariates = ['mean_lagged_return','var_lagged_return','volume%','ma_close', 'macd', 'rsi']

model = CausalModel(
    data=df,
    treatment='gap_up',
    outcome='daily_return',
    common_causes=covariates
)

# 识别和估计因果效应
identified_estimand = model.identify_effect()
estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression",
    test_significance=True
    
)
print(estimate)

linear_regression
Estimation failed! No relevant identified estimand available for this estimation method.


# 结果和原文不一样
原文当中是两者有显著相关关系，所以可能是因为原始数据变化了？还是前面自己 ta 的计算出现了问题？
*** Causal Estimate ***

## Identified estimand
Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
   d                                                                          
────────(E[daily_return|mean_lagged_return,macd,var_lagged_return,volume%,ma_c
d[gapᵤₚ]                                                                      


lose,rsi])

Estimand assumption 1, Unconfoundedness: If U→{gap_up} and U→daily_return then P(daily_return|gap_up,mean_lagged_return,macd,var_lagged_return,volume%,ma_close,rsi,U) = P(daily_return|gap_up,mean_lagged_return,macd,var_lagged_return,volume%,ma_close,rsi)

## Realized estimand
b: daily_return~gap_up+mean_lagged_return+macd+var_lagged_return+volume%+ma_close+rsi
Target units: ate

## Estimate
Mean value: 0.48177222629125094
p-value: [5.90117295e-24]

In [13]:
# Placebo测试 安慰剂检验

placebo_test = model.refute_estimate(identified_estimand, estimate, method_name="placebo_treatment_refuter", placebo_type="permute")

print(placebo_test)

ValueError: Aborting refutation! No valid estimate is provided.

In [ ]:
import dowhy
from dowhy import CausalModel

# 定义因果模型
model = CausalModel(
        data=data,
        treatment='X', 
        outcome='Y',
        graph='graph.dot')

SyntaxError: invalid character in identifier (<ipython-input-14-5dd6b6afa74b>, line 1)

In [15]:
from dowhy import CausalModel
import dowhy.datasets

# Load some sample data
data = dowhy.datasets.linear_dataset(
    beta=10,
    num_common_causes=5,
    num_instruments=2,
    num_samples=10000,
    treatment_is_binary=True)

In [19]:
# I. Create a causal model from the data and given graph.
model = CausalModel(
    data=data["df"],
    treatment=data["treatment_name"],
    outcome=data["outcome_name"],
    graph=data["gml_graph"])  # Or alternatively, as nx.DiGraph

# II. Identify causal effect and return target estimands
identified_estimand = model.identify_effect()

# III. Estimate the target estimand using a statistical method.
estimate = model.estimate_effect(identified_estimand,
                                 method_name="backdoor.propensity_score_matching")

# IV. Refute the obtained estimate using multiple robustness checks.
refute_results = model.refute_estimate(identified_estimand, estimate,
                                       method_name="random_common_cause")

propensity_score_matching


/Users/wangshuaibo/anaconda3/envs/ShuaiGaitpy/lib/python3.6/site-packages/sklearn/utils/validation.py:724: DataConversionWarning:

A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().

/Users/wangshuaibo/anaconda3/envs/ShuaiGaitpy/lib/python3.6/site-packages/sklearn/utils/validation.py:724: DataConversionWarning:

A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().

/Users/wangshuaibo/anaconda3/envs/ShuaiGaitpy/lib/python3.6/site-packages/sklearn/utils/validation.py:724: DataConversionWarning:

A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().

/Users/wangshuaibo/anaconda3/envs/ShuaiGaitpy/lib/python3.6/site-packages/sklearn/utils/validation.py:724: DataConversionWarning:

A column-vector y was passed when a 1d array was expected. Please change t